In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_FILE = "results/summary.csv"

CLASSIFICATION = [
"cmc",
"connect-4",
"credit-g",
"kc1",
"splice",
"vehicle",
"phoneme",
"eucalyptus",
"eye_movements",
"house_16H",
"Bank_Customer_Churn_Dataset"
]

REGRESSION = [
"elevators",
"socmob",
"pol"
]

MODELS = ["mlp","xgboost","randomforest"]


########################################
# LOAD
########################################

df = pd.read_csv(RESULTS_FILE)

df["method"] = df["detector"].astype(str) + "_" + df["treatment"]


########################################
# IMPROVEMENT
########################################

def compute_improvement(group):

    baseline = group[
        (group.detector.isna()) |
        (group.detector=="None")
    ]

    if len(baseline)==0:
        return group

    base = baseline.iloc[0]["mean"]

    if group.iloc[0]["dataset"] in REGRESSION:

        group["improvement"] = base - group["mean"]

    else:

        group["improvement"] = group["mean"] - base

    return group


df = df.groupby(
["dataset","model"],
group_keys=False
).apply(compute_improvement)


########################################
# TABLE BUILDER
########################################

def build_raw_table(data,model):

    m = data[data.model==model]

    pivot = m.pivot_table(

        index="dataset",

        columns="method",

        values=["mean","std","outlier_pct"],

        aggfunc="first"

    )

    table = pd.DataFrame(index=pivot.index)

    for method in m["method"].unique():

        try:

            mean = pivot["mean"][method]

            std = pivot["std"][method]

            out = pivot["outlier_pct"][method]

            table[method] = (

                mean.round(4).astype(str)
                + " ± "
                + std.round(4).astype(str)
                + " ("
                + (100*out).round(2).astype(str)
                + "%)"
            )

        except:
            table[method]="NA"

    return table


########################################
# IMPROVEMENT TABLE
########################################

def build_improvement_table(data,model):

    m = data[data.model==model]

    pivot = m.pivot_table(

        index="dataset",

        columns="method",

        values=["improvement","std","outlier_pct"],

        aggfunc="first"

    )

    table = pd.DataFrame(index=pivot.index)

    for method in m["method"].unique():

        try:

            imp = pivot["improvement"][method]

            std = pivot["std"][method]

            out = pivot["outlier_pct"][method]

            table[method] = (

                imp.round(4).astype(str)
                + " ± "
                + std.round(4).astype(str)
                + " ("
                + (100*out).round(2).astype(str)
                + "%)"
            )

        except:

            table[method]="NA"

    avg = m.groupby("method")["improvement"].mean()

    table.loc["AVERAGE"] = avg

    return table


########################################
# METHOD SUMMARY
########################################

def method_summary(data):

    rows=[]

    for method in data.method.unique():

        sub = data[data.method==method]

        improved = (sub.improvement>0).sum()

        total = len(sub)

        avg_imp = sub.improvement.mean()

        std_imp = sub.improvement.std()

        avg_out = sub.outlier_pct.mean()

        corr = sub[["improvement","outlier_pct"]].corr().iloc[0,1]

        rows.append({

            "method":method,

            "improved":

            f"{improved}/{total}",

            "avg_improvement":avg_imp,

            "std_improvement":std_imp,

            "avg_outliers":avg_out,

            "correlation":corr

        })

    return pd.DataFrame(rows)


########################################
# SCATTER
########################################

def scatter(data,model,folder):

    m=data[data.model==model]

    plt.figure(figsize=(8,6))

    for method in m.method.unique():

        sub=m[m.method==method]

        plt.scatter(

            sub.outlier_pct*100,

            sub.improvement,

            label=method

        )

    plt.axhline(0,color="black")

    plt.xlabel("Outliers %")

    plt.ylabel("Performance change")

    plt.title(model)

    plt.legend(fontsize=7)

    plt.savefig(

        f"{folder}/{model}_scatter.png",

        dpi=200,

        bbox_inches="tight"

    )

    plt.close()


########################################
# MAIN EXPORT
########################################

def export(group,name):

    base=f"analysis/{name}"

    os.makedirs(base+"/tables",exist_ok=True)

    os.makedirs(base+"/plots",exist_ok=True)

    for model in MODELS:

        raw=build_raw_table(group,model)

        raw.to_csv(

            f"{base}/tables/{model}_raw.csv"
        )

        imp=build_improvement_table(group,model)

        imp.to_csv(

            f"{base}/tables/{model}_improvement.csv"
        )

        scatter(group,model,base+"/plots")

    summary=method_summary(group)

    summary.to_csv(

        f"{base}/tables/method_summary.csv",

        index=False
    )


########################################
# RUN
########################################

export(
df[df.dataset.isin(CLASSIFICATION)],
"classification"
)

export(
df[df.dataset.isin(REGRESSION)],
"regression"
)

print("Analysis complete")